# Phase 2 Feedback TFIM-QRC v2.1 Compact-Summary Probe

The previous v2 probe used sequential dense/feedback QRC dynamics but exported the full observable trajectory, producing 580 readout features. That likely over-expanded noisy temporal dynamics.

This v2.1 probe keeps the sequential dense/feedback dynamics but compresses the trajectory into:

- final full observable vector;
- temporal mean/std of full observables;
- final/mean/std/min/max memory-Z summaries;
- final/mean/std/min/max readout feedback summaries.

Goal: test temporal memory without dumping the full 20-step trajectory into Ridge.

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.qrc.feedback_tfim_reservoir import (
    FeedbackTFIMQRCConfig,
    diagnose_reservoir_feature_splits,
    fit_feedback_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_feedback_qrc_result,
)

## 1. Data and PCA-6 sequence windows

In [2]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

sequence_splits_6 = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

display(pca6.explained_variance)
print({name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_6.items()})

,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.408573,0.408573
1,2,0.112421,0.520994
2,3,0.090849,0.611843
3,4,0.073259,0.685102
4,5,0.064206,0.749308
5,6,0.055584,0.804892


{'train': ((5420, 40, 6), (5420,)), 'val': ((1219, 40, 6), (1219,)), 'test': ((1019, 40, 6), (1019,))}


## 2. Compact-summary feedback sweep

Same v2 dynamics, but `feature_collection="summary"`.

In [3]:
summary_rows = []
summary_diag_rows = []
summary_results = {}

for feedback_gain in [0.0, 0.3, 0.7]:
    config = FeedbackTFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        temporal_steps=20,
        temporal_policy="even",
        input_qubits=(0, 1),
        memory_qubits=(2, 3, 4),
        readout_qubits=(5,),
        observable_mode="zxzz",
        feature_collection="summary",
        trotter_steps_per_time=1,
        evolution_time=0.25,
        input_scale=3.141592653589793 / 2,
        transverse_field=0.5,
        input_memory_coupling_scale=1.2,
        memory_coupling_scale=1.0,
        readout_coupling_scale=0.7,
        weak_background_coupling_scale=0.15,
        feedback_gain=float(feedback_gain),
        feedback_rotation="rz",
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        disorder_strength=0.10,
    )

    run_name = f"feedback_tfim_v21_summary_gain_{feedback_gain}"
    print(f"Running {run_name}")

    result = fit_feedback_tfim_qrc_regressor(
        sequence_splits_6,
        config=config,
        target=target,
        verbose=True,
    )

    summary_results[run_name] = result

    row = summarize_feedback_qrc_result(result)
    row["run_name"] = run_name
    summary_rows.append(row)

    _, y_train, _ = sequence_splits_6["train"]
    _, y_val, _ = sequence_splits_6["val"]
    _, y_test, _ = sequence_splits_6["test"]

    diag = diagnose_reservoir_feature_splits(
        result.train_features,
        result.val_features,
        result.test_features,
        y_train,
        y_val,
        y_test,
    )
    diag.insert(0, "run_name", run_name)
    summary_diag_rows.append(diag)

summary_table = pd.DataFrame(summary_rows)
summary_diagnostics = pd.concat(summary_diag_rows, ignore_index=True)

Running feedback_tfim_v21_summary_gain_0.0
Feedback-QRC sample 0/5420
Feedback-QRC sample 250/5420
Feedback-QRC sample 500/5420
Feedback-QRC sample 750/5420
Feedback-QRC sample 1000/5420
Feedback-QRC sample 1250/5420
Feedback-QRC sample 1500/5420
Feedback-QRC sample 1750/5420
Feedback-QRC sample 2000/5420
Feedback-QRC sample 2250/5420
Feedback-QRC sample 2500/5420
Feedback-QRC sample 2750/5420
Feedback-QRC sample 3000/5420
Feedback-QRC sample 3250/5420
Feedback-QRC sample 3500/5420
Feedback-QRC sample 3750/5420
Feedback-QRC sample 4000/5420
Feedback-QRC sample 4250/5420
Feedback-QRC sample 4500/5420
Feedback-QRC sample 4750/5420
Feedback-QRC sample 5000/5420
Feedback-QRC sample 5250/5420
Feedback-QRC sample 0/1219
Feedback-QRC sample 250/1219
Feedback-QRC sample 500/1219
Feedback-QRC sample 750/1219
Feedback-QRC sample 1000/1219
Feedback-QRC sample 0/1019
Feedback-QRC sample 250/1019
Feedback-QRC sample 500/1019
Feedback-QRC sample 750/1019
Feedback-QRC sample 1000/1019
Running feedbac

## 3. Metrics

In [4]:
metric_cols = [
    "run_name",
    "feature_collection",
    "feedback_gain",
    "temporal_steps",
    "n_reservoir_features",
    "train_rmse",
    "val_rmse",
    "test_rmse",
    "train_qlike",
    "val_qlike",
    "test_qlike",
    "train_mz_r2",
    "val_mz_r2",
    "test_mz_r2",
]

summary_table[metric_cols].sort_values("test_rmse")

,run_name,feature_collection,feedback_gain,temporal_steps,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
0,feedback_tfim_v21_summary_gain_0.0,summary,0.0,20,107,0.093501,0.063741,0.106632,-2.302991,-3.014493,-1.806623,0.160287,0.044733,0.027318
1,feedback_tfim_v21_summary_gain_0.3,summary,0.3,20,107,0.093628,0.063951,0.106659,-2.298900,-3.015256,-1.779389,0.160090,0.043374,0.025974
2,feedback_tfim_v21_summary_gain_0.7,summary,0.7,20,107,0.092956,0.063373,0.107033,-2.309279,-3.019648,-1.752455,0.169845,0.053467,0.021777


## 4. Diagnostics

In [5]:
diagnostic_cols = [
    "run_name",
    "split",
    "n_samples",
    "n_features",
    "near_constant_features",
    "feature_std_min",
    "feature_std_median",
    "feature_std_max",
    "effective_rank",
    "condition_number",
    "mean_abs_feature_target_corr",
    "max_abs_feature_target_corr",
    "mean_abs_shift_vs_train",
    "max_abs_shift_vs_train",
]

summary_diagnostics[diagnostic_cols]

,run_name,split,n_samples,n_features,near_constant_features,feature_std_min,feature_std_median,feature_std_max,effective_rank,condition_number,mean_abs_feature_target_corr,max_abs_feature_target_corr,mean_abs_shift_vs_train,max_abs_shift_vs_train
0,feedback_tfim_v21_summary_gain_0.0,train,5420,107,0,0.004084,0.095543,0.397359,27.794622,2593.572459,0.135969,0.232938,0.000000,0.000000
1,feedback_tfim_v21_summary_gain_0.0,val,1219,107,0,0.004518,0.094323,0.418940,25.739007,3040.056542,0.108928,0.259193,0.194347,0.456464
2,feedback_tfim_v21_summary_gain_0.0,test,1019,107,0,0.004368,0.097642,0.400139,27.419360,2836.632986,0.086100,0.266800,0.051601,0.410228
3,feedback_tfim_v21_summary_gain_0.3,train,5420,107,0,0.004626,0.084205,0.431983,24.860067,3208.657337,0.124886,0.225730,0.000000,0.000000
4,feedback_tfim_v21_summary_gain_0.3,val,1219,107,0,0.005099,0.078387,0.454222,22.976032,3904.394928,0.104195,0.246739,0.207857,0.452975
5,feedback_tfim_v21_summary_gain_0.3,test,1019,107,0,0.004984,0.078944,0.444333,24.002799,3386.005172,0.090405,0.256860,0.068932,0.388365
6,feedback_tfim_v21_summary_gain_0.7,train,5420,107,0,0.005104,0.063007,0.466798,22.211006,3902.206553,0.145741,0.280307,0.000000,0.000000
7,feedback_tfim_v21_summary_gain_0.7,val,1219,107,0,0.005640,0.061794,0.481548,20.813923,4458.006016,0.123623,0.239032,0.199272,0.444668
8,feedback_tfim_v21_summary_gain_0.7,test,1019,107,0,0.005429,0.063716,0.474980,21.562669,4310.960741,0.139352,0.235833,0.050018,0.394332


## 5. Save outputs

In [6]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

summary_table.to_csv(out_dir / "phase2_feedback_tfim_qrc_v21_summary_probe.csv", index=False)
summary_diagnostics.to_csv(out_dir / "phase2_feedback_tfim_qrc_v21_summary_diagnostics.csv", index=False)

print("Saved v2.1 compact-summary outputs to", out_dir)

Saved v2.1 compact-summary outputs to results/tables


## 6. Interpretation rule

Compare against:

```text
v1 best static QRC:
RMSE  = 0.102618
QLIKE = -1.942716
MZ R² = 0.072134

v2 full-trajectory no-feedback:
RMSE  = 0.107123
QLIKE = -1.802335
MZ R² = 0.015167
```

Question: does compact temporal memory rescue v2 from the 580-feature overexpansion?